# 03 — Main 1D CNN Pipeline

Canonical thesis pipeline: wavelet denoising, adaptive volatility labeling, walk-forward CNN training.

Core logic lives in `src/fqw/`. This notebook is a thin orchestrator.

In [ ]:
# !pip install -e ..

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt

from fqw.config import CNNConfig, DEFAULT_TICKERS
from fqw.data.loading import load_ticker_5min, preprocess_ticker
from fqw.training import run_backtest_with_fine_tuning
from fqw.backtest import analyze_trades, analyze_strategy_performance
from fqw.experiments import run_batch_backtest_to_csv

pd.set_option("display.max_columns", None)
cfg = CNNConfig(data_dir="../data")
print("Tickers:", DEFAULT_TICKERS)

## Single ticker example

In [ ]:
TICKER = "YDEX"
ALPHA = 1.0
CONFIDENCE = 0.75

df = load_ticker_5min(TICKER, data_dir=cfg.data_dir)
df = preprocess_ticker(df, use_wavelet=True, use_technical_indicators=False)

preds, metrics = run_backtest_with_fine_tuning(
    df, TICKER, alpha=ALPHA, confidence_threshold=CONFIDENCE, config=cfg
)
display(preds.head())
trades = analyze_trades(preds, commission=cfg.commission, rf_annual=cfg.rf_annual)
trades

## Batch grid search (thesis experiments)

Reproduces the main results table. Outputs CSV to `results/`.

In [ ]:
for alpha, conf in cfg.param_pairs:
    run_batch_backtest_to_csv(
        tickers=cfg.tickers,
        alpha=alpha,
        confidence_threshold=conf,
        config=cfg,
        filename=f"results/cnn_alpha_{alpha}_conf_{conf}.csv",
        use_wavelet=True,
        use_technical_indicators=False,
    )